In [15]:
import sys
import torchvision.transforms.functional as F

sys.modules["torchvision.transforms.functional_tensor"] = F

In [18]:
import os
import json
from typing import Optional, List, Tuple

import numpy as np
import torch
from PIL import Image
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler
from diffusers.utils.torch_utils import randn_tensor

import t2v_metrics


# =========================================================
# 1) Загрузка базовых компонентов Stable Diffusion 1.5
# =========================================================
MODEL_ID = "runwayml/stable-diffusion-v1-5"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder", torch_dtype=dtype
).to(device)
vae = AutoencoderKL.from_pretrained(
    MODEL_ID, subfolder="vae", torch_dtype=dtype
).to(device)
unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID, subfolder="unet", torch_dtype=dtype
).to(device)
scheduler = PNDMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

text_encoder.eval()
vae.eval()
unet.eval()


# =========================================================
# 2) Загрузка VLM scorer
# =========================================================
def build_vlm_scorer(device: Optional[str] = None, model_name: str = "clip-flant5-xl"):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    return t2v_metrics.VQAScore(model=model_name, device=device)


vqa_scorer = build_vlm_scorer(device=device, model_name="clip-flant5-xl")


# =========================================================
# 3) Хелперы
# =========================================================
@torch.no_grad()
def encode_prompt(prompt: str, negative_prompt: str = "", max_length: int = 77):
    text_inputs = tokenizer(
        prompt,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    uncond_inputs = tokenizer(
        negative_prompt,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    cond_embeds = text_encoder(text_inputs.input_ids.to(device))[0]
    uncond_embeds = text_encoder(uncond_inputs.input_ids.to(device))[0]

    return cond_embeds, uncond_embeds


@torch.no_grad()
def decode_latents(latents: torch.Tensor) -> torch.Tensor:
    """
    latents: [B, 4, H/8, W/8]
    returns: images [B, 3, H, W] in [0,1]
    """
    vae_dtype = next(vae.parameters()).dtype
    latents = (latents / 0.18215).to(device=vae.device, dtype=vae_dtype)

    images = vae.decode(latents).sample
    images = (images / 2 + 0.5).clamp(0, 1)
    return images.float()


def save_image_tensor(images_bchw: torch.Tensor, path: str):
    """
    images_bchw: [B,3,H,W] in [0,1]
    """
    image = images_bchw[0].detach().cpu().clamp(0, 1)
    image = (image.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    Image.fromarray(image).save(path)



@torch.no_grad()
def compute_vlm_score_from_image(
    image: torch.Tensor,
    prompt: str,
    vqa_scorer,
) -> float:
    """
    image:
      - [3, H, W] или [1, 3, H, W]
      - значения в [0, 1]

    prompt:
      строка prompt

    Возвращает scalar VLM score.
    """
    if image.ndim == 3:
        image = image.unsqueeze(0)

    if image.ndim != 4 or image.shape[1] != 3:
        raise ValueError(f"Ожидался image tensor [1,3,H,W] или [3,H,W], получено {tuple(image.shape)}")

    model_device = getattr(vqa_scorer, "device", device)

    image = image.to(model_device, dtype=torch.float32)
    detail = [prompt]

    scores = vqa_scorer(image, detail)

    if isinstance(scores, torch.Tensor):
        return float(scores.squeeze().detach().cpu().item())

    return float(np.array(scores).squeeze())


@torch.no_grad()
def compute_vlm_score_from_latents(
    latents: torch.Tensor,
    prompt: str,
    vqa_scorer,
) -> float:
    image = decode_latents(latents)   # [1,3,H,W] in [0,1]
    return compute_vlm_score_from_image(
        image=image,
        prompt=prompt,
        vqa_scorer=vqa_scorer,
    )




def save_scores_json(scores: List[dict], path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(scores, f, ensure_ascii=False, indent=2)


# =========================================================
# 4) Основная генерация:
#    - сохраняет декодированные промежуточные шаги
#    - считает VLM score на каждом шаге
#    - сохраняет метаданные в JSON
# =========================================================
@torch.no_grad()
def generate_with_step_vlm_scores(
    prompt: str,
    negative_prompt: str = "",
    height: int = 512,
    width: int = 512,
    num_inference_steps: int = 30,
    guidance_scale: float = 7.5,
    seed: Optional[int] = 0,
    save_dir: str = "sd15_run",
    save_every: int = 1,
    vqa_scorer=None,
    question_template: Optional[str] = None,
    answer_template: Optional[str] = None,
) -> Tuple[torch.Tensor, List[dict]]:
    """
    Возвращает:
      final_images: [1,3,H,W] in [0,1]
      step_records: список словарей со score по шагам
    """
    assert height % 8 == 0 and width % 8 == 0, "height/width должны делиться на 8"

    if vqa_scorer is None:
        raise ValueError("Нужно передать vqa_scorer")

    os.makedirs(save_dir, exist_ok=True)
    steps_dir = os.path.join(save_dir, "steps")
    os.makedirs(steps_dir, exist_ok=True)

    cond_embeds, uncond_embeds = encode_prompt(prompt, negative_prompt)
    text_embeds = torch.cat([uncond_embeds, cond_embeds], dim=0)

    scheduler.set_timesteps(num_inference_steps, device=device)

    batch_size = 1
    latent_shape = (batch_size, unet.config.in_channels, height // 8, width // 8)

    generator = None
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)

    latents = randn_tensor(latent_shape, generator=generator, device=device, dtype=dtype)
    latents = latents * scheduler.init_noise_sigma

    n_digits = max(4, len(str(len(scheduler.timesteps))))
    step_records = []

    for i, t in enumerate(scheduler.timesteps):
        latent_model_input = torch.cat([latents] * 2, dim=0)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        noise_pred = unet(
            latent_model_input,
            t,
            encoder_hidden_states=text_embeds,
        ).sample

        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        latents = scheduler.step(noise_pred, t, latents).prev_sample

        should_save = (save_every > 0 and i % save_every == 0) or (i == len(scheduler.timesteps) - 1)
        if should_save:
            decoded_image = decode_latents(latents)

            score = compute_vlm_score_from_image(
                image=decoded_image,
                prompt=prompt,
                vqa_scorer=vqa_scorer,
                question_template=question_template,
                answer_template=answer_template,
            )

            t_int = int(t.item()) if hasattr(t, "item") else int(t)
            filename = f"step_{i:0{n_digits}d}_t{t_int}.png"
            image_path = os.path.join(steps_dir, filename)
            save_image_tensor(decoded_image, image_path)

            record = {
                "step_index": int(i),
                "timestep": t_int,
                "image_path": image_path,
                "vlm_score": float(score),
            }
            step_records.append(record)

            print(f"step={i:03d} | t={t_int:04d} | vlm_score={score:.6f} | saved={image_path}")

    final_images = decode_latents(latents)
    final_score = compute_vlm_score_from_image(
        image=final_images,
        prompt=prompt,
        vqa_scorer=vqa_scorer,
        question_template=question_template,
        answer_template=answer_template,
    )

    final_image_path = os.path.join(save_dir, "final.png")
    save_image_tensor(final_images, final_image_path)

    meta = {
        "prompt": prompt,
        "negative_prompt": negative_prompt,
        "height": height,
        "width": width,
        "num_inference_steps": num_inference_steps,
        "guidance_scale": guidance_scale,
        "seed": seed,
        "final_image_path": final_image_path,
        "final_vlm_score": float(final_score),
        "steps": step_records,
    }

    save_scores_json(meta, os.path.join(save_dir, "scores.json"))

    print(f"\nFinal image saved to: {final_image_path}")
    print(f"Final VLM score: {final_score:.6f}")
    print(f"Scores JSON saved to: {os.path.join(save_dir, 'scores.json')}")

    return final_images, step_records


# =========================================================
# 5) Дополнительная удобная функция:
#    score для уже готового декодированного изображения
# =========================================================
@torch.no_grad()
def vlm_score_for_decoded_sample(
    image: torch.Tensor,
    prompt: str,
    vqa_scorer=None,
) -> float:
    if vqa_scorer is None:
        raise ValueError("Нужно передать vqa_scorer")

    return compute_vlm_score_from_image(
        image=image,
        prompt=prompt,
        vqa_scorer=vqa_scorer,
    )

# =========================================================
# 6) Пример использования
# =========================================================
if __name__ == "__main__":
    prompt = "a cozy small library room, warm light, ultra detailed, 35mm photo"
    negative_prompt = "low quality, blurry, bad anatomy"

    final_images, step_scores = generate_with_step_vlm_scores(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=512,
        width=512,
        num_inference_steps=30,
        guidance_scale=7.5,
        seed=42,
        save_dir="sd15_vlm_run",
        save_every=1,
        vqa_scorer=vqa_scorer,
        # при желании можно явно задать шаблоны:
        # question_template='Does this figure show "{}"? Please answer yes or no.',
        # answer_template='Yes',
    )

    final_score = vlm_score_for_decoded_sample(
        image=final_images,
        prompt=prompt,
        vqa_scorer=vqa_scorer,
    )

    print(f"\nRecomputed final VLM score: {final_score:.6f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 58.00 MiB. GPU 0 has a total capacity of 15.73 GiB of which 50.62 MiB is free. Process 2037151 has 15.67 GiB memory in use. Of the allocated memory 15.05 GiB is allocated by PyTorch, and 426.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [19]:
!pip freeze > requitements.txt

In [ ]:
print(1)

1


In [13]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip install xformers
!pip install t2v-metrics
!pip install ninja packaging
!pip install flash-attn --no-build-isolation

In [10]:
!pip install pytorchvideo

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188758 sha256=556aeaaee609b0e3abee178e6216c0c4283c33fdf07954e4fa6d65987fc57302
  Stored in directory: /root/.cache/pip/wheels/ff/4e/81/0f72a543be9ed7eb737c95bfc5da4025e73226b44368074ece
Successfully built pytorchvideo
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pytorchvideo]


In [15]:
import sys
!{sys.executable} -m pip uninstall -y transformers t2v-metrics
!{sys.executable} -m pip install --no-cache-dir "transformers==4.49.0" "t2v-metrics"

Found existing installation: transformers 4.37.2
Uninstalling transformers-4.37.2:
  Successfully uninstalled transformers-4.37.2
Found existing installation: t2v_metrics 3.0
Uninstalling t2v_metrics-3.0:
  Successfully uninstalled t2v_metrics-3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 47.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 248.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 83.6 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 81.6 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 141.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 89.9 MB/s  0:00:07m0:00:0100:01
 

In [16]:
import importlib.util
print(importlib.util.find_spec("transformers.models.paligemma.modeling_paligemma"))

None


In [ ]:
import transformers
print(transformers.__version__)

from transformers.models.paligemma.modeling_paligemma import PaliGemmaForConditionalGeneration
import t2v_metrics
print("ok")

In [ ]:
print(1)